In [7]:
!pip install -U transformers accelerate pandas

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"   # swap this for each rung of your ladder

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)

In [ ]:
import pandas as pd

grants = pd.DataFrame([
    {"title": "Women TechEU 2026–2028",
     "text": "Women TechEU 2026–2028. Grant + support: €75,000 non-dilutive funding plus mentoring/coaching. Deadline 2026-09-16. Active pathway; full-proposal cutoff reported."},
    {"title": "Tara's Circle 2026",
     "text": "Tara's Circle 2026. Prize / cash awards: €100,000 total pool; €10k/€6k/€4k awards across categories. Deadline 2026-09-17. Open as last reported."},
    {"title": "Her Agenda Breakthrough Grant",
     "text": "Her Agenda Breakthrough Grant. Grant: $5,000. Deadline 2026-09-18. Open as last reported."},
])

In [ ]:
import json, re

PROMPT = """Extract information from the grant description below.
Return ONLY a JSON object with exactly these keys:
"funding_type", "amount_total", "currency", "deadline" (YYYY-MM-DD), "target_applicants", "focus_area", "geography".
Use null for anything not explicitly stated. Do not guess.

Grant description:
{text}"""

def extract(text, max_new_tokens=1024):
    # DeepSeek-R1 guidance: no system prompt, put all instructions in the user turn
    messages = [{"role": "user", "content": PROMPT.format(text=text)}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.6, top_p=0.95,
        )
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # Reasoning models emit a chain of thought first; keep only what follows </think>
    answer = raw.split("</think>")[-1] if "</think>" in raw else raw
    match = re.search(r"\{.*\}", answer, re.DOTALL)
    if not match:
        return {"_error": "no_json", "_raw": raw[-300:]}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return {"_error": "invalid_json", "_raw": match.group(0)[:300]}

In [5]:
records = []
for _, g in grants.iterrows():
    result = extract(g["text"])
    records.append({"title": g["title"], **result})

extracted = pd.DataFrame(records)
valid = extracted["_error"].isna().mean() if "_error" in extracted else 1.0
print(f"JSON validity: {valid:.0%}")
extracted

JSON validity: 100%


,title,funding_type,amount_total,currency,deadline,target_applicants,focus_area,geography
0,Women TechEU 2026–2028,grant,75000,euros,2026-09-16,None,Women TechEU,None
1,Tara's Circle 2026,Prize / cash awards,"€100,000",€,2026-09-17,None,NaN,None
2,Her Agenda Breakthrough Grant,Her Agenda Breakthrough Grant,None,NaN,2026-09-18,None,NaN,None


In [6]:
gold = {
    "Women TechEU 2026–2028": {"amount_total": 75000, "currency": "EUR", "deadline": "2026-09-16"},
}

def field_accuracy(extracted_df, gold):
    hits, total = 0, 0
    for _, row in extracted_df.iterrows():
        for field, truth in gold.get(row["title"], {}).items():
            total += 1
            hits += str(row.get(field)) == str(truth)
    return hits / total if total else None

print("Field accuracy:", field_accuracy(extracted, gold))

Field accuracy: 0.6666666666666666
